# Lista 3

## Numpy podstawy

(8pkt + 2pkt)

Na liście znajduje się 8 zadań do wykonania w trakcie zajęć. Po rozwiązaniu zadania pokaż kod prowadzącemu i odpowiedz na **pytanie kontrolne** (są one podane przy każdym zadaniu) — wtedy przyznajemy punkt. Dodatkowo prześlij zadanie na platformie skos.

Jeśli w poleceniu nie jest wskazane użycie pętli `for` - nie korzystaj z niej.

Zadania domowe są oznaczone 🏠. To zadania o numerach 9 i 10.

Kazde zadnanie warte jest 1pkt.

## 1. Analiza pikseli obrazu

**Kontekst:** W AI obrazy to tablice NumPy (np. `H×W×C`). Proste operacje na obrazach można robić wektorowo bez pętli.

**Polecenie**

* Utwórz przykładową tablicę 3×3 obrazu RGB zawierającą wartości 0–255.
* Zwiększ jasność o 10% — ale **nie przekraczaj 255** (`np.clip`).
* Wypisz wynik.

**Podpowiedź**
Zwiększ jasność obrazu mnożąc każde pasmo RGB przez 110%. 

In [9]:
import numpy as np

np.random.seed(1)

A = np.random.randint(0, 256, size=(3, 3), dtype=np.uint8)

bright = np.clip(np.round(A * 1.1), 0, 255).astype(np.uint8)

print("Oryginał:\n", A)
print("Po zwiększeniu jasności o 10%:\n", bright)

Oryginał:
 [[ 37 244 193]
 [106 235 128]
 [ 71 255 140]]
Po zwiększeniu jasności o 10%:
 [[ 41 255 212]
 [117 255 141]
 [ 78 255 154]]


## 2. Normalizacja danych wejściowych

**Kontekst:** Dane wejściowe do modeli zazwyczaj normujemy — albo do `[0,1]`, albo standaryzujemy do średniej 0 i odchylenia 1.

**Polecenie**

* Wygeneruj 1D tablicę 100 losowych liczb w przedziale `0..255`.
* Przeskaluj na `[0,1]`.
* Następnie stwórz znormalizowaną wersję z średnią 0 i odchyleniem 1 (standaryzacja).
* Nie używaj pętli.

**Pytanie kontrolne:** Co się stanie, gdy `std == 0`?

In [12]:
import numpy as np

x = np.random.random(100) * 255

# skalowanie do [0,1]
x_scaled = (x - np.min(x)) / (np.max(x) - np.min(x))

# standaryzacja: (x - mean)/std
mean = np.mean(x)
std = np.std(x)
x_standardized = (x - mean) / std

print("mean scaled:", np.mean(x_scaled))
print("mean standardized:", np.mean(x_standardized))
print("std standardized:", np.std(x_standardized))

mean scaled: 0.508405836129779
mean standardized: -2.0983215165415458e-16
std standardized: 0.9999999999999999


## 3. Wycinanie fragmentów danych

**Kontekst:** Często dzielimy długi sygnał na krótsze okna (np. do ekstrakcji cech).

**Polecenie**

* Stwórz 1D tablicę długości 1000 (np. `np.arange(1000)`).
* Podziel ją na 10 fragmentów po 100 elementów (`reshape`).
* Oblicz średnią każdego fragmentu (wektor 10-elementowy).

**Pytanie kontrolne:** Co policzymy jeśli obliczymy `axis=0`?

In [14]:
import numpy as np

x = np.arange(1000)
chunks = x.reshape(10,100)   # shape (10,100)
means = np.mean(chunks, axis=1)   # średnia wzdłuż kolumn -> 10 wartości

print("shape chunks:", chunks.shape)
print("means:", means)

shape chunks: (10, 100)
means: [ 49.5 149.5 249.5 349.5 449.5 549.5 649.5 749.5 849.5 949.5]


## 4. Operacje wzdłuż osi

**Kontekst:** Przy normalizacjach i agregacjach często operujemy wzdłuż konkretnej osi.

**Polecenie**

* Utwórz `X = np.random.randint(0, 100, (5,4))`.
* Oblicz:

  * min każdego wiersza używając `X.min(...)`
  * max każdej kolumny używając `X.max(...)`
* Znormalizuj każdy wiersz tak, aby suma elementów w wierszu była równa 1. Sprawdź poprawność rozwiązania sumując wartości w każdym wierszu.

In [21]:
import numpy as np

X = np.random.randint(0, 100, (5,4))

row_mins = np.min(X, axis=1)
col_maxs = np.max(X, axis=0)

# normalizacja wierszy do sumy 1
row_sums = np.sum(X, axis=1, keepdims=True)   # shape (5,1) - jakiego parametru trzeba użyć?
X_norm = X / row_sums

print("X:\n", X)
print("row_mins:", row_mins)
print("col_maxs:", col_maxs)
print("row_sums:", row_sums)
print("X_norm (suma wiersza):", X_norm.sum(axis=1))

X:
 [[78 17 39 35]
 [81 28 22 38]
 [41 74 77 70]
 [25 48 50 62]
 [44 54  0 86]]
row_mins: [17 22 41 25  0]
col_maxs: [81 74 77 86]
row_sums: [[169]
 [169]
 [262]
 [185]
 [184]]
X_norm (suma wiersza): [1. 1. 1. 1. 1.]


## 5. Analiza embeddingów słów

**Kontekst:** W NLP słowa reprezentujemy wektorami (embeddingi). Często chcemy średnią reprezentację zdania i podobieństwo kosinusowe.

**Polecenie**

* Stwórz macierz `E` o kształcie `(5,50)` — 5 embeddingów po 50 wymiarów (`np.random.randn`).
* Oblicz:

  * `mean_embedding` (średni wektor po wierszach)
  * macierz podobieństw kosinusowych między wszystkimi parami wektorów (5×5)

**Pytanie kontrolne:** Jak interpretujesz wartości w `cosine_sim` (zakres i znaczenie)?

---

### Wskazówka

Zastanów się, jak można obliczyć **podobieństwo kosinusowe** dwóch wektorów `a` i `b`, jeśli wiesz, że:

$$
\cos(\theta) = \frac{a \cdot b}{|a| \cdot |b|}
$$

Spróbuj najpierw policzyć to dla jednej pary wektorów w NumPy,
używając `np.dot()` i `np.linalg.norm()`.

A następnie pomyśl:

* Jak znormalizować wszystkie wektory jednocześnie, żeby każdy miał długość 1?
* Jak wykorzystać mnożenie macierzy (`@`), żeby w jednym kroku dostać podobieństwa między wszystkimi parami wektorów (5×5)?

In [23]:
import numpy as np

E = np.random.randn(5, 50)

# średni wektor (embedding zdania)
mean_embedding = np.mean(E, axis=0)

# normalizacja wektorów do długości 1
norms = np.linalg.norm(E, axis=1, keepdims=True)
E_normalized = E / norms

# macierz podobieństw kosinusowych = E_norm @ E_norm.T
cosine_sim = E_normalized @ E_normalized.T

print("mean_embedding shape:", mean_embedding.shape)
print("cosine_sim shape:", cosine_sim.shape)
print("cosine_sim (przykład):\n", cosine_sim)

mean_embedding shape: (50,)
cosine_sim shape: (5, 5)
cosine_sim (przykład):
 [[ 1.         -0.33236349  0.15989604 -0.0258956  -0.04760108]
 [-0.33236349  1.         -0.2051065  -0.03914381 -0.01284699]
 [ 0.15989604 -0.2051065   1.         -0.16022838  0.01343825]
 [-0.0258956  -0.03914381 -0.16022838  1.         -0.11356754]
 [-0.04760108 -0.01284699  0.01343825 -0.11356754  1.        ]]


## 6. Sklejanie sensorów — dane z pojazdu autonomicznego

**Kontekst:** Dane z różnych sensorów różnią się formatem — trzeba je połączyć w jedną macierz cech.

**Polecenie**

* Stwórz:

  ```python
  lidar = np.random.rand(100, 2)   # x,y # lidar - Light Detection and Ranging
  camera = np.random.rand(100, 1)  # jasność
  radar = np.random.rand(100, 1)   # prędkość
  ```
* Połącz je kolumnowo w macierz `X` o kształcie `(100,4)`.
* Podziel `X` na `train` (80%) i `test` (20%) (np. `np.array_split` lub indeksowanie).
* Oblicz średnią jasność i średnią prędkość dla obu zestawów.

**Pytanie kontrolne:** Dlaczego warto przetasować dane (shuffle) przed splitem w prawdziwym pipeline?

In [28]:
import numpy as np

np.random.seed(1)

lidar = np.random.rand(100, 2)   # x,y
camera = np.random.rand(100, 1)  # jasność
radar = np.random.rand(100, 1)   # prędkość

X = np.hstack((lidar, camera, radar))
assert X.shape == (100, 4)

# split train/test (bez shuffle dla prostoty)
n_train = int(0.8 * len(X))
train, test = X[:n_train], X[n_train:]

# indeksy kolumn: camera -> 2, radar -> 3 (0-based)
mean_camera_train = np.mean(train[:, 2])
mean_radar_train = np.mean(train[:, 3])
mean_camera_test = np.mean(test[:, 2])
mean_radar_test = np.mean(test[:, 3])

print("X.shape:", X.shape)
print("train/test shapes:", train.shape, test.shape)
print("mean camera train/test:", mean_camera_train, mean_camera_test)
print("mean radar train/test:", mean_radar_train, mean_radar_test)

X.shape: (100, 4)
train/test shapes: (80, 4) (20, 4)
mean camera train/test: 0.5581777190030272 0.5408440810754731
mean radar train/test: 0.5194488688865528 0.47276907127944884


## 7. NumPy vs. czysty Python

**Kontekst:** NumPy działa o wiele szybciej niż pętle Pythonowe dla operacji numerycznych. Tu wykorzystamy `np.linspace`.

**Polecenie**

* Stwórz `x = np.linspace(0, 1, 10_000_000)`.
* Oblicz `s = sin(x)**2 + cos(x)**2`:

  * a) w czystym Pythonie (pętla `for` i `math.sin`, `math.cos`)
  * b) w NumPy (`np.sin`, `np.cos` — wektoryzacja)
* Porównaj czasy wykonania (wskazane: `%timeit` w notebooku lub `time.time()`).

In [32]:
import numpy as np
import math
import time

# dane
x = np.linspace(0, 1, 10_000_000)

# NumPy version
t0 = time.time()
y_np = np.sin(x)**2 + np.cos(x)**2
t_np = time.time() - t0

# Python loop version (bardzo wolne; uruchamiaj ostrożnie; jak trwa dłuej ni 15s, to przerwij i zmniejsz przykład)
t0 = time.time()
s = 0.0
for xi in x:           
    s += math.sin(xi)**2 + math.cos(xi)**2
t_py = time.time() - t0

print(np.isclose(y_np.sum(), s))
print(f"NumPy time: {t_np:.3f}s")
print(f"Pure Python loop time: {t_py:.3f}s")

True
NumPy time: 0.158s
Pure Python loop time: 2.141s


## 8. Copy, View

**Kontekst:** Podczas preprocessingu zespoły czasami pracują na tym samym obiekcie myśląc, że mają kopię — to prowadzi do skażenia danych.

**Polecenie**

* Stwórz `patients = np.random.randint(60, 120, (5, 4))`.
  * `doctors_view`, gdzie doktor będzie przeglądał tylko co drugi pomiar,
  * `research_copy`, czyli kopię patients.
* Lekarz przez pomyłkę wykona `doctors_view[:] = -1`.
* Sprawdź, czy `patients` się zmieniło i czy `research_copy` pozostało bez zmian. Wyjaśnij.

**Pytanie kontrolne:** W jaki sposób zabezpieczyć workflow, żeby przypadkowe zmiany nie wpływały na bazę danych?

In [33]:
import numpy as np

patients = np.random.randint(60, 120, (5, 4))
doctors_view = patients[:, ::2].view()
research_copy = patients.copy()

print("przed zmiana:\n", patients)
doctors_view[:] = -1
print("po zmianie doctors_view:\n", patients)
print("research_copy (czy zmienione?):\n", research_copy)

# wykrywamy, które są view / copy
print("doctors_view.base is patients?", doctors_view.base is patients)
print("research_copy.base is patients?", research_copy.base is patients)

przed zmiana:
 [[ 67 108  88  79]
 [ 87 103  79  75]
 [ 66  84 118  82]
 [ 71  97  61  89]
 [ 75 111  94  82]]
po zmianie doctors_view:
 [[ -1 108  -1  79]
 [ -1 103  -1  75]
 [ -1  84  -1  82]
 [ -1  97  -1  89]
 [ -1 111  -1  82]]
research_copy (czy zmienione?):
 [[ 67 108  88  79]
 [ 87 103  79  75]
 [ 66  84 118  82]
 [ 71  97  61  89]
 [ 75 111  94  82]]
doctors_view.base is patients? True
research_copy.base is patients? False


## 9. 🏠 Zaawansowana augmentacja i analiza obrazów RGB

**Kontekst:** Przy trenowaniu CNN operujemy na obrazach RGB. Często robimy augmentacje (jasność, kontrast, rotacje, przesunięcia) i porównujemy obrazy np. za pomocą **cosine similarity**. Trzeba uważać, które operacje zwracają **widok**, a które **kopię**, aby nie psuć danych przez przypadek.

**Polecenie**

1. Wygeneruj **10 losowych obrazów RGB**, każdy o wymiarach `32×32×3` z wartościami `uint8` (0–255).
2. Napisz funkcję `cosine_similarity(img1, img2)`, która oblicza podobieństwo kosinusowe między dwoma obrazami traktowanymi jako wektory.
3. Stwórz funkcję `augment(img)` która tworzy warianty obrazu:

   * oryginał,
   * flip poziomy,
   * flip pionowy,
   * rotacja 90°,
   * jaśniejsza wersja,
   * wersja z większym kontrastem,
   * mały przesunięty fragment (crop + pad),
   * zamiana kanałów RGB na BGR.
4. Dla każdego wariantu sprawdź, które są **widokiem**, a które **kopią** (`.base`).
5. Oblicz macierz podobieństwa kosinusowego między wszystkimi 10 obrazami oryginalnymi i wypisz średnie podobieństwo.
6. Oblicz podobieństwo między oryginałami a ich wariantami i sprawdź, które operacje najbardziej zmieniają obraz (najmniejsze cosine similarity).   

In [ ]:
import numpy as np

images = np.random.randint(0, 256, size=(10, 32, 32, 3), dtype=np.uint8)

def cosine_similarity(img1: np.ndarray, img2: np.ndarray):
    img1 = img1.ravel().astype(float)
    img2 = img2.ravel().astype(float)
    return (img1 * img2).sum() / (np.linalg.norm(img1) * np.linalg.norm(img2))

def augment(img: np.ndarray):
    augment_img = {}
    augment_img['original'] = img.copy()

    # flip poziomy
    augment_img['horizontal_flip'] = img[:, ::-1, :].copy()

    # flip pinowy
    augment_img['vertical_flip'] = img[::-1, ...].copy()

    # rotacja 90 stopni
    augment_img['rotation_90'] = np.rot90(img)

    # jaśniejsza wersja
    augment_img['bright'] = np.clip(np.round(img * 1.1), 0, 255).astype(np.uint8)
    # wersja z wiekszym kontrastem

    # maly przesutniety fragment (crop + pad)

    # zamiana kanalow
    augment_img['swap_colors'] = img[..., ::-1].copy()

    return augment_img





## 10. 🏠 Gra w życie (slicing + maski)

**Kontekst:** *Gra w życie* Conwaya to automat komórkowy: plansza złożona z komórek żywych (`1`) i martwych (`0`), która w kolejnych krokach zmienia się według prostych reguł. Naiwna implementacja to dwie zagnieżdżone pętle po wszystkich komórkach, ale w NumPy całą generację policzysz bez pętli po komórkach, używając przesuniętych wycinków i masek.

**Polecenie**

Napisz funkcję `step(board)`, która dla tablicy 0/1 zwraca następną generację. Reguły:

* żywa komórka z **2 lub 3** żywymi sąsiadami przeżywa, w przeciwnym razie umiera,
* martwa komórka z **dokładnie 3** żywymi sąsiadami ożywa,
* komórki poza planszą traktujemy jako martwe.

Sąsiadów jest 8 (poziomo, pionowo i po przekątnych). Zadanie wykonaj w krokach:

1. Zrób tablicę `P` o rozmiarze `(wiersze + 2, kolumny + 2)` wypełnioną zerami i wstaw do jej środka `board`. Dodatkowa ramka zer załatwia krawędzie planszy.
2. Policz tablicę `n` z liczbą żywych sąsiadów każdej komórki, **sumując 8 przesuniętych wycinków** `P` (każdy ma rozmiar `board`). Pętla `for` jest dozwolona tylko do przejścia po tych 8 przesunięciach.
3. Wyznacz nową planszę za pomocą masek logicznych (bez pętli po komórkach).
4. Sprawdź funkcję na „szybowcu” z kodu poniżej: po 4 generacjach powinien wyglądać tak samo jak na początku, ale przesunięty o jedną komórkę w dół i w prawo.

**Wskazówka:** maski logiczne łączymy operatorami `&` (i) oraz `|` (lub), a każde porównanie trzeba ująć w nawiasy, np. `(n == 3) | (board == 1) & (n == 2)`. Wynik maski zamienisz na `0/1` przez `.astype(int)`.

**Pytanie kontrolne:** Dlaczego tablicę `n` tworzysz jako nową (`np.zeros_like`), a nie jako `n = P[1:-1, 1:-1]`, do której dodajesz kolejne wycinki? Co by się stało z `P`?

In [ ]:
import numpy as np

board = np.zeros((20, 20), dtype=int)
board[1, 2] = board[2, 3] = board[3, 1] = board[3, 2] = board[3, 3] = 1   # szybowiec

def step(board):
    # Uzupełnij
    return # Uzupełnij

b = board
for _ in range(4):          # pętla po generacjach jest dozwolona
    b = step(b)

print("żywych komórek:", b.sum())
print("szybowiec przesunięty o (1, 1)?", np.array_equal(b[1:, 1:], board[:-1, :-1]))